### Transformer Encoder, Decoder를 활용한 문장 번역

<br>
translator_ko_en_train_v2.csv : 증강된 학습 데이터<br>

### 파일을 Google Drive에 저장하고 사용

개인 Drive 폴더 구조에 맞게 변경하여 사용 필요

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Colab Notebooks/Transformer/TF05')

print(f'현재 작업 폴더: {os.getcwd()}')

Mounted at /content/drive
현재 작업 폴더: /content/drive/MyDrive/Colab Notebooks/Transformer/TF05


In [ ]:
!pip install sacrebleu

import io
import math
from pathlib import Path

import pandas as pd
import sacrebleu
import sentencepiece as spm
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 14.2 MB/s eta 0:00:00


In [ ]:
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
BOS_TOKEN = "<bos>"  # begin of sentence, decoder 시작 토큰
EOS_TOKEN = "<eos>"  # end of sentence, 생성 종료 토큰

In [ ]:
# 실험 설정. 실험마다 이 값을 직접 바꿔서 실행한다.
# 경로는 src 디렉터리 기준 상대경로
PARAMS = {
    "train_path": "../data/translator_ko_en_train_v2.csv.zip",
    "val_path": "../data/translator_ko_en_val_v2.csv.zip",
    "src_vocab_size": 12000,
    "tgt_vocab_size": 16000,
    "sentencepiece_model_type": "bpe",
    "sentencepiece_character_coverage": 1.0,
    "sentencepiece_byte_fallback": True,
    "sentencepiece_split_digits": True,
    "sentencepiece_normalization_rule_name": "nmt_nfkc",
    "sentencepiece_num_threads": 8,
    "max_src_len": 192,
    "max_tgt_len": 160,
    "batch_size": 64,
    "epochs": 50,
    "num_workers": 1,
    "pin_memory": True,
    "num_encoder_layers": 6,
    "num_decoder_layers": 6,
    "nhead": 8,
    "d_model": 512,
    "dim_feedforward": 2048,
    "learning_rate": 1e-4,
    "weight_decay": 1e-3,
    "label_smoothing": 0.1,
    "dropout": 0.1,
    "grad_clip": 1.0,  # None
    # 데이터가 baseline(244k행)의 약 10배라 warmup도 10배로 늘려 epoch 기준 LR 곡선을 baseline과 맞춤
    "warmup_steps": 40000,
    "beam_size": 4,
    "length_penalty_alpha": 0.4,
    "max_eval_samples": 1000,
    "sample_count": 5,
    # best model 경로. 이어서 학습용 마지막 상태는 같은 폴더의 <이름>_last.pt 에 저장
    "output_path": "../model/translator_ko_to_en_v2.pt",
    "tensorboard_dir": "../tensorboard/translator_ko_to_en",
    "seed": 42,
}

In [ ]:
class SentencePieceTokenizer:
    def __init__(
            self,
            vocab_size,
            model_type="bpe",
            character_coverage=1.0,  # 학습 데이터에 존재하는 문자를 최대한 모두 포함
            byte_fallback=True,  # vocab에 없는 문자를 UTF-8 byte 토큰으로 분리하여 <unk> 방지
            split_digits=True,  # 긴 숫자를 분리해 희귀한 숫자 표현에 대응
            # 전각 문자를 일반 문자로 정규화 해주는 SentencePiece 기본 룰. 예) ＡＢＣ → ABC
            normalization_rule_name="nmt_nfkc",
            num_threads=8,  # SentencePiece 학습에 사용할 CPU 스레드 수
    ):
        self.vocab_size = vocab_size
        self.model_type = model_type
        self.character_coverage = character_coverage
        self.byte_fallback = byte_fallback
        self.split_digits = split_digits
        self.normalization_rule_name = normalization_rule_name
        self.num_threads = num_threads
        self.model_proto = None
        self.processor = None

    @property
    def pad_id(self):
        return self.processor.pad_id()

    @property
    def unk_id(self):
        return self.processor.unk_id()

    @property
    def bos_id(self):
        return self.processor.bos_id()

    @property
    def eos_id(self):
        return self.processor.eos_id()

    def __len__(self):  # vocab 크기를 반환
        return self.processor.get_piece_size()

    def _load_model(self, model_proto):  # model_proto : 학습된 SentencePiece 모델 전체를 담은 바이너리 데이터
        self.model_proto = bytes(model_proto)
        self.processor = spm.SentencePieceProcessor()
        self.processor.load_from_serialized_proto(self.model_proto)

    def build_vocab(self, texts):
        model_writer = io.BytesIO()  # 학습된 SentencePiece 모델을 가상 바이너리 파일에 저장
        sentence_iterator = (  # 학습 문장을 한 문장씩 SentencePiece에 전달하는 용도
            text
            for text in (str(text).strip() for text in texts)
            if text
        )
        spm.SentencePieceTrainer.train(
            sentence_iterator=sentence_iterator,
            model_writer=model_writer,
            vocab_size=self.vocab_size,
            model_type=self.model_type,
            character_coverage=self.character_coverage,
            byte_fallback=self.byte_fallback,
            split_digits=self.split_digits,
            normalization_rule_name=self.normalization_rule_name,
            pad_id=0,
            unk_id=1,
            bos_id=2,
            eos_id=3,
            pad_piece=PAD_TOKEN,
            unk_piece=UNK_TOKEN,
            bos_piece=BOS_TOKEN,
            eos_piece=EOS_TOKEN,
            hard_vocab_limit=False,  # 지정한 vocab 크기를 반드시 정확하게 채우도록 강제하지 않는 설정
            num_threads=self.num_threads,
        )
        self._load_model(model_writer.getvalue())  # tokenizer 학습 결과를 조회하여 세팅

    def encode(self, text, max_len, add_bos=True, add_eos=True):
        ids = self.processor.encode(str(text).strip(), out_type=int)
        if add_bos:
            ids = [self.bos_id] + ids
        if add_eos:
            ids = ids + [self.eos_id]
        # ["<bos>", "I", "like", "coffee", "<eos>"]

        if len(ids) > max_len:
            ids = ids[:max_len]
            if add_eos:
                ids[-1] = self.eos_id  # 잘라도 마지막은 EOS 토큰으로 대체
        return ids

    def decode(self, ids, remove_special=True):
        filtered_ids = []
        for idx in ids:
            idx = int(idx)
            if idx < 0 or idx >= len(self):  # ID가 vocab 범위 밖이면 <unk>로 처리
                idx = self.unk_id
            if remove_special:
                if idx in {self.pad_id, self.bos_id}:  # <pad>, <bos>는 출력 문장에 보이지 않게 Skip
                    continue
                if idx == self.eos_id:  # <eos>를 만나면 문장 생성을 끝냄
                    break
            filtered_ids.append(idx)
        return self.processor.decode(filtered_ids).strip()

    def to_dict(self):
        return {
            "vocab_size": self.vocab_size,
            "model_type": self.model_type,
            "character_coverage": self.character_coverage,
            "byte_fallback": self.byte_fallback,
            "split_digits": self.split_digits,
            "normalization_rule_name": self.normalization_rule_name,
            "num_threads": self.num_threads,
            "model_proto": self.model_proto,
        }

    @classmethod
    def from_dict(cls, data):  # 저장해둔 dict에서 tokenizer를 복원
        tokenizer = cls(
            vocab_size=data["vocab_size"],
            model_type=data.get("model_type", "bpe"),
            character_coverage=data.get("character_coverage", 1.0),
            byte_fallback=data.get("byte_fallback", True),
            split_digits=data.get("split_digits", True),
            normalization_rule_name=data.get(
                "normalization_rule_name", "nmt_nfkc"
            ),
            num_threads=data.get("num_threads", 8),
        )
        tokenizer._load_model(data["model_proto"])
        return tokenizer

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, df, src_tokenizer, tgt_tokenizer, max_src_len=64, max_tgt_len=64):
        self.src_texts = df["src"].astype(str).tolist()
        self.target_texts = df["target"].astype(str).tolist()
        self.src_tokenizer = src_tokenizer
        self.tgt_tokenizer = tgt_tokenizer
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len

    def __len__(self):  # Dataset 크기 (샘플 개수) 반환
        return len(self.src_texts)

    def __getitem__(self, idx):
        src_text = self.src_texts[idx]
        target_text = self.target_texts[idx]

        src_ids = self.src_tokenizer.encode(src_text, self.max_src_len)
        target_ids = self.tgt_tokenizer.encode(target_text, self.max_tgt_len)  # [2, 4, 5, 6, 3]

        # teacher forcing을 통해 모델이 다음 단어를 학습할 수 있도록 토큰을 Shift 하여 준비
        # 입력: <bos> I drink coffee
        # 정답: I drink coffee <eos>
        return {
            "src_ids": src_ids,
            "tgt_input_ids": target_ids[:-1],  # [2, 4, 5, 6] : <bos> I drink coffee
            "tgt_output_ids": target_ids[1:],  # [4, 5, 6, 3] : I drink coffee <eos>
            "src_text": src_text,
            "target_text": target_text,
        }

In [ ]:
# 각 문장의 토큰 길이가 다르니 Batch 내에서 가장 긴 길이를 기준으로 맞춤. 단, 빈 공간은 <pad> 세팅
# sequences = [
#     [2, 10, 11, 3],
#     [2, 20, 3],
# ]
# batch = [
#     [2, 10, 11, 3],
#     [2, 20, 3, 0],
# ]
def pad_sequence(sequences, pad_id):
    max_len = max(len(seq) for seq in sequences)  # 가장 긴 토큰의 길이
    batch = torch.full((len(sequences), max_len), pad_id, dtype=torch.long)  # <pad> ID로 채운 tensor 준비
    for idx, seq in enumerate(sequences):  # 각 문장의 토큰 길이만큼 업데이트
        batch[idx, : len(seq)] = torch.tensor(seq, dtype=torch.long)
    return batch

In [ ]:
# collate_fn() : Dataset 리스트를 모아서 모델에서 사용할 각각의 리스트로 재구성
# 이 함수를 만들어 리턴하는 함수
def make_collate_fn(src_pad_id, tgt_pad_id):
    def collate_fn(batch):
        src_ids = pad_sequence([item["src_ids"] for item in batch], src_pad_id)
        tgt_input_ids = pad_sequence([item["tgt_input_ids"] for item in batch], tgt_pad_id)
        tgt_output_ids = pad_sequence([item["tgt_output_ids"] for item in batch], tgt_pad_id)
        return {
            "src_ids": src_ids,
            "tgt_input_ids": tgt_input_ids,
            "tgt_output_ids": tgt_output_ids,
            "src_text": [item["src_text"] for item in batch],
            "target_text": [item["target_text"] for item in batch],
        }

    return collate_fn

In [ ]:
# PositionalEncoding 값은 고정값이나 nn.Module을 상속하여 만들면 아래의 장점이 있음
# 1) model.to(device) 때 같이 이동가능. 일반 tensor로 사용시에는 별도로 GPU로 이동 시켜줘야 함
# 2) register_buffer()를 통해 model.state_dict()에 포함되어 저장가능
#  parameter: 학습됨, optimizer가 업데이트
#  buffer: 학습 안 됨, 그래도 모델 상태로 저장됨
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        # torch.arange(max_len) : tensor([0, 1, 2, 3])
        # unsqueeze(1) : tensor([[0], [1], [2], [3]]), shape : (max_len, 1)
        position = torch.arange(max_len).unsqueeze(1)

        # 논문의 식
        # PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
        # PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
        # 예) pos=3, d_model=8, 아래 값을 sin, cos에 넣어 계산
        # k = 0 -> 1 / 10000^(0/8) = 1
        # k = 2 -> 1 / 10000^(2/8) = 1 / 10 = 0.1
        # k = 4 -> 1 / 10000^(4/8) = 1 / 100 = 0.01
        # k = 6 -> 1 / 10000^(6/8) = 1 / 1000 = 0.001

        # 그런데 sin, cos 분모 부분(div_term)을 아래와 같이 역수로 변환하여 사용
        # 곱셈 형태가 tensor 연산에서 자연스럽고 빨라 아래와 같이 역수를 준비하고 position에 곱함
        # 10000^(-2i/d_model) = exp(log(10000^(-2i/d_model)))
        # = exp((-2i/d_model) * log(10000)) = exp(2i * (-log(10000) / d_model))

        # torch.arange(0, 8, 2) : tensor([0, 2, 4, 6]), 위 수식의 2i를 한번에 준비
        # div_term = [1.0, 0.1, 0.01, 0.001]
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(max_len, d_model)

        # 아래 position * div_term을 sin, cos에 넣어 계산한 값을 사용
        # [[0, 0.0, 0.00, 0.000],
        #  [1, 0.1, 0.01, 0.001],
        #  [2, 0.2, 0.02, 0.002],
        #  [3, 0.3, 0.03, 0.003]]
        pe[:, 0::2] = torch.sin(position * div_term)  # 0::2 -> 0부터 시작해서 2칸씩 건너뜀
        pe[:, 1::2] = torch.cos(position * div_term)

        # pe shape:              (max_len, d_model)
        # pe.unsqueeze(0) shape: (1, max_len, d_model), batch로 계산하는것을 고려하여 차원 추가
        # register_buffer() : 학습 parameter는 아니지만 모델에 딸린 tensor로 지정
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        # 입력 x의 sequence 길이. shape이 (batch_size, seq_len, d_model) 라서 1번째
        seq_len = x.size(1)

        # 입력 embedding에 해당 길이만큼 positional encoding을 더함
        # x: (32, 20, 256)라면, self.pe[:, :seq_len] (1, 20, 256) 이 각 batch 마다 동일하게 더해짐
        x = x + self.pe[:, :seq_len]

        return self.dropout(x)  # 과적합 방지용 dropout 추가

In [ ]:
class TranslationTransformer(nn.Module):
    def __init__(
            self,
            src_vocab_size,
            tgt_vocab_size,
            src_pad_id,
            tgt_pad_id,
            d_model=256,
            nhead=4,
            num_encoder_layers=3,
            num_decoder_layers=3,
            dim_feedforward=512,
            max_len=128,
            dropout=0.1,
    ):
        super().__init__()
        self.d_model = d_model
        self.src_pad_id = src_pad_id
        self.tgt_pad_id = tgt_pad_id

        # nn.Embedding : 정수 입력을 받아 정해진 차원의 벡터를 리턴 하는 Lookup Table
        # padding_idx : Lookup Table 학습할 필요 없어 제외
        self.src_embedding = nn.Embedding(src_vocab_size, d_model, padding_idx=src_pad_id)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model, padding_idx=tgt_pad_id)
        self.pos_encoding = PositionalEncoding(d_model, max_len=max_len, dropout=dropout)
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )
        self.generator = nn.Linear(d_model, tgt_vocab_size)

    def _generate_square_subsequent_mask(self, seq_len, device):
        # bool 타입의 Causal Mask 생성 (True면 마스킹 됨. 보지 못하게 막음)
        # [[False,  True,  True],
        #  [False, False,  True],
        #  [False, False, False]]
        # 0번 위치는 1,2번 미래를 못 봄
        # Mask를 활용하면 모델 학습시 문장을 한 번에 학습 할 수 있는 장점이 있음
        return torch.triu(torch.ones((seq_len, seq_len), dtype=torch.bool, device=device), diagonal=1)

    def forward(self, src_ids, tgt_input_ids):
        # src에서 <pad> 위치를 찾음. True인 곳은 attention에서 무시
        # src_ids = [
        #     [2, 10, 11, 3],
        #     [2, 20, 3, 0],
        # ]
        # src_key_padding_mask = [
        #     [False, False, False, False],
        #     [False, False, False, True],
        # ]
        src_key_padding_mask = src_ids.eq(self.src_pad_id)
        tgt_key_padding_mask = tgt_input_ids.eq(self.tgt_pad_id)
        # target 길이에 맞춰 causal mask 준비
        tgt_mask = self._generate_square_subsequent_mask(tgt_input_ids.size(1), tgt_input_ids.device)

        # embedding vector에서 token id에 해당하는 것 추출하고 scaling 적용
        # d_model=256 이면 16을 곱하게 됨
        # token embedding의 scale을 키워서 positional encoding에 너무 묻히지 않게 하려는 의도
        src = self.src_embedding(src_ids) * math.sqrt(self.d_model)
        tgt = self.tgt_embedding(tgt_input_ids) * math.sqrt(self.d_model)
        src = self.pos_encoding(src)
        tgt = self.pos_encoding(tgt)

        output = self.transformer(
            src,
            tgt,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask,
        )
        return self.generator(output)

In [ ]:
def count_trainable_parameters(model):
    # 학습되는 parameter만 Count. shape = [30000, 256] 이면 numel = 30000 * 256 = 7680000
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

In [ ]:
def get_transformer_lr_lambda(warmup_steps):
    # Noam-style warmup scheduler에 쓸 lambda 함수 생성
    # warmup_steps 까지 서서히 올렸다가 이후 서서히 줄임
    def lr_lambda(step):
        step = max(1, step)
        if step <= warmup_steps:
            return step / warmup_steps
        return math.sqrt(warmup_steps / step)

    return lr_lambda

In [ ]:
def train_epoch(model, loader, criterion, optimizer, scheduler, device, grad_clip=1.0):
    model.train()
    total_loss = 0.0
    total_tokens = 0

    for batch in loader:
        src_ids = batch["src_ids"].to(device)
        tgt_input_ids = batch["tgt_input_ids"].to(device)
        tgt_output_ids = batch["tgt_output_ids"].to(device)

        optimizer.zero_grad()  # Gradient를 0으로 초기화. 하지 않으면 미분값이 누적되어 더해짐
        logits = model(src_ids, tgt_input_ids)

        loss_sum = criterion(
            # nn.CrossEntropyLoss에서 요구하는 shape로 변경
            # logits.size(-1) : 텐서의 마지막 차원이 길이. 여기서는 vocab_size
            # logits.reshape(-1, vocab_size) : 마지막 차원만 유지하고 나머지 차원 통합 (batch_size * sequence, vocab_size)
            logits.reshape(-1, logits.size(-1)),
            tgt_output_ids.reshape(-1),
        )

        # <pad>가 아닌 실제 target token 개수 count
        # clamp_min(1) : 전부 pad인 이상한 batch에서도 0으로 나누지 않기 위한 방어 코드
        token_count = tgt_output_ids.ne(model.tgt_pad_id).sum().clamp_min(1)

        # batch마다 문장 길이와 padding 개수가 다르기 때문에 token당 평균 loss 계산 필요
        # 미적용시 긴 문장이 많은 batch는 gradient가 커지고, 짧은 문장이 많은 batch는 gradient가 작아져 안정적이지 못함
        loss = loss_sum / token_count
        loss.backward()

        # gradient clipping. Transformer 학습 중 gradient가 너무 커져서 불안정해지는 것을 막음
        if grad_clip is not None and grad_clip > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        scheduler.step()

        total_loss += float(loss_sum.item())
        total_tokens += int(token_count.item())

    return total_loss / max(1, total_tokens)

In [ ]:
def eval_loss(model, loader, criterion, device):
    model.eval()  # drop out, batch normalization 적용 부분을 평가 모드로 전환
    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():  # Chain Rule 계산을 위해 어떤 연산이 적용 되었는지 저장하는데, 이것을 안하겠다는 의미
        # 평가시에는 미분 backward()를 안하므로 메모리 절약을 위해 사용
        # 기본적으로 net(inputs) 실행시마다 매번 새롭게 grad_fn을 구성하기 때문에 이를 생략할 수 있음

        for batch in loader:
            src_ids = batch["src_ids"].to(device)
            tgt_input_ids = batch["tgt_input_ids"].to(device)
            tgt_output_ids = batch["tgt_output_ids"].to(device)

            logits = model(src_ids, tgt_input_ids)
            loss_sum = criterion(
                logits.reshape(-1, logits.size(-1)),
                tgt_output_ids.reshape(-1),
            )
            token_count = tgt_output_ids.ne(model.tgt_pad_id).sum().clamp_min(1)
            total_loss += float(loss_sum.item())
            total_tokens += int(token_count.item())

    return total_loss / max(1, total_tokens)

In [ ]:
def get_length_penalty(generated_length, alpha):
    # GNMT/Transformer에서 사용한 길이 정규화. alpha=0이면 길이 보정을 적용하지 않음.
    return ((5.0 + max(1, generated_length)) / 6.0) ** alpha

In [ ]:
# 문장 하나를 실제 번역처럼 생성. beam_size=1이면 greedy decoding과 동일하게 동작.
def translate_sentence(
        model,
        src_text,
        src_tokenizer,
        tgt_tokenizer,
        device,
        max_src_len=64,
        max_tgt_len=64,
        beam_size=1,
        length_penalty_alpha=0.0,
):
    model.eval()
    src_ids = src_tokenizer.encode(src_text, max_src_len)  # src_ids = [2, 10, 11, 3], shape = (4)
    src_tensor = torch.tensor([src_ids], dtype=torch.long, device=device)  # src_tensor shape = (1, 4)
    # (생성 token ID 목록, 누적 log 확률, EOS 생성 여부)
    beams = [([tgt_tokenizer.bos_id], 0.0, False)]

    with torch.no_grad():
        for _ in range(max_tgt_len - 1):
            finished_beams = [beam for beam in beams if beam[2]]
            active_beams = [beam for beam in beams if not beam[2]]
            if not active_beams:
                break

            # 같은 길이의 활성 beam들을 한 배치로 처리해 후보 token 확률을 계산
            tgt_tensor = torch.tensor(
                [generated for generated, _, _ in active_beams],
                dtype=torch.long,
                device=device,
            )
            src_batch = src_tensor.expand(len(active_beams), -1)
            logits = model(src_batch, tgt_tensor)
            log_probs = torch.log_softmax(logits[:, -1, :], dim=-1)

            candidates = finished_beams
            for beam_index, (generated, score, _) in enumerate(active_beams):
                top_log_probs, top_ids = log_probs[beam_index].topk(beam_size)
                for token_log_prob, token_id in zip(top_log_probs.tolist(), top_ids.tolist()):
                    next_id = int(token_id)
                    candidates.append(
                        (
                            generated + [next_id],
                            score + float(token_log_prob),
                            next_id == tgt_tokenizer.eos_id,
                        )
                    )

            # 누적 log 확률에 길이 정규화를 적용해 다음 단계에서 유지할 beam을 선정
            beams = sorted(
                candidates,
                key=lambda beam: beam[1] / get_length_penalty(
                    len(beam[0]) - 1,
                    length_penalty_alpha,
                ),
                reverse=True,
            )[:beam_size]

        completed_beams = [beam for beam in beams if beam[2]]
        best_beam = max(
            completed_beams or beams,
            key=lambda beam: beam[1] / get_length_penalty(
                len(beam[0]) - 1,
                length_penalty_alpha,
            ),
        )

    return tgt_tokenizer.decode(best_beam[0])

In [ ]:
def compute_bleu_chrf(hypotheses, references):  # sacrebleu 활용하여 BLEU, chrF를 계산
    bleu = sacrebleu.corpus_bleu(hypotheses, [references]).score
    chrf = sacrebleu.corpus_chrf(hypotheses, [references]).score
    return bleu, chrf

In [ ]:
def evaluate_translation(
        model,
        loss_loader,
        metric_loader,
        criterion,
        src_tokenizer,
        tgt_tokenizer,
        device,
        max_src_len,
        max_tgt_len,
        beam_size,
        length_penalty_alpha,
        sample_count=5,
):
    loss = eval_loss(model, loss_loader, criterion, device)

    hypotheses = []
    references = []
    sources = []

    for batch in metric_loader:
        for src_text, target_text in zip(batch["src_text"], batch["target_text"]):
            hyp = translate_sentence(
                model,
                src_text,
                src_tokenizer,
                tgt_tokenizer,
                device,
                max_src_len=max_src_len,
                max_tgt_len=max_tgt_len,
                beam_size=beam_size,
                length_penalty_alpha=length_penalty_alpha,
            )
            sources.append(src_text)
            references.append(target_text)
            hypotheses.append(hyp)

    bleu, chrf = compute_bleu_chrf(hypotheses, references)

    # 출력용 샘플 추출
    # [("나는 커피를 마신다", "I drink coffee", "I drink coffee"),
    #  ("오늘 날씨가 좋다", "The weather is nice today", "Today weather is good")]
    samples = list(zip(sources, references, hypotheses))[:sample_count]
    return loss, bleu, chrf, samples

In [ ]:
def read_parallel_csv(path):  # src, target 컬럼을 가진 CSV를 읽음
    # 확장자가 .zip이면 pandas가 자동으로 압축 해제하여 읽는다 (내부에 파일이 하나뿐이어야 함)
    df = pd.read_csv(path, sep=",", encoding="utf-8")
    missing = [col for col in ["src", "target"] if col not in df.columns]
    if missing:  # 필요한 컬럼이 없으면 에러
        raise ValueError(f"Missing columns in {path}: {missing}")

    df = df[["src", "target"]].copy()  # 필요한 컬럼만 추출
    df = df.dropna()  # 결측치가 있는 행 제거
    df["src"] = df["src"].astype(str).str.strip()  # 문자열로 바꾸고 앞뒤 공백을 제거
    df["target"] = df["target"].astype(str).str.strip()
    df = df[(df["src"] != "") & (df["target"] != "")]  # 빈 문장 제거
    return df.reset_index(drop=True)  # 인덱스를 0부터 다시 정리해서 반환

In [ ]:
def sample_eval_dataframe(val_df, max_eval_samples, seed):
    if max_eval_samples is None or max_eval_samples <= 0 or max_eval_samples >= len(val_df):
        return val_df.reset_index(drop=True)

    return val_df.sample(
        n=max_eval_samples,
        random_state=seed,
    ).reset_index(drop=True)

In [ ]:
def percentile(sorted_values, ratio):  # 정렬된 숫자 리스트에서 대략적인 분위수 값을 꺼냄
    if not sorted_values:
        return 0
    index = int((len(sorted_values) - 1) * ratio)
    return sorted_values[index]

In [ ]:
def analyze_tokenizer(name, tokenizer, texts, max_len):  # tokenizer build 결과를 출력
    # +2 : <bos>, <eos>
    token_lengths = sorted(len(tokenizer.processor.encode(text, out_type=int)) + 2 for text in texts)

    # max_len보다 긴 문장. 잘린 문장 확인용
    truncated_count = sum(length > max_len for length in token_lengths)
    truncated_ratio = truncated_count / max(1, len(token_lengths))

    print(f"\n{name} tokenizer")
    print("-" * 72)
    print(f"vocab size: {len(tokenizer):,}")
    print(
        f"special ids: pad={tokenizer.pad_id}, unk={tokenizer.unk_id}, bos={tokenizer.bos_id}, eos={tokenizer.eos_id}"
    )
    print(f"max_len setting: {max_len}")
    print(f"max observed length (+bos/eos): {max(token_lengths, default=0)}")
    avg_len = sum(token_lengths) / max(1, len(token_lengths))
    print(f"avg observed length (+bos/eos): {avg_len:.2f}")
    print(f"p95 observed length (+bos/eos): {percentile(token_lengths, 0.95)}")
    print(f"truncated by max_len: {truncated_count:,} ({truncated_ratio:.2%})")

In [ ]:
def create_sentencepiece_tokenizer(vocab_size, params):
    return SentencePieceTokenizer(
        vocab_size=vocab_size,
        model_type=params["sentencepiece_model_type"],
        character_coverage=params["sentencepiece_character_coverage"],
        byte_fallback=params["sentencepiece_byte_fallback"],
        split_digits=params["sentencepiece_split_digits"],
        normalization_rule_name=params["sentencepiece_normalization_rule_name"],
        num_threads=params["sentencepiece_num_threads"],
    )

In [ ]:
def build_tokenizers(train_df, params):
    src_texts = train_df["src"].tolist()
    tgt_texts = train_df["target"].tolist()

    src_tokenizer = create_sentencepiece_tokenizer(
        vocab_size=params["src_vocab_size"],
        params=params,
    )
    tgt_tokenizer = create_sentencepiece_tokenizer(
        vocab_size=params["tgt_vocab_size"],
        params=params,
    )
    print("\nBuilding source SentencePiece tokenizer")
    src_tokenizer.build_vocab(src_texts)
    print("Building target SentencePiece tokenizer")
    tgt_tokenizer.build_vocab(tgt_texts)

    analyze_tokenizer("Source", src_tokenizer, src_texts, params["max_src_len"])
    analyze_tokenizer("Target", tgt_tokenizer, tgt_texts, params["max_tgt_len"])

    return src_tokenizer, tgt_tokenizer

In [ ]:
def prepare_data_loaders(params):
    train_df = read_parallel_csv(params["train_path"])
    val_df = read_parallel_csv(params["val_path"])

    print(f"Train rows: {len(train_df):,}")
    print(f"Validation rows: {len(val_df):,}")
    metric_val_df = sample_eval_dataframe(
        val_df,
        max_eval_samples=params["max_eval_samples"],
        seed=params["seed"],
    )
    print(f"Metric evaluation rows: {len(metric_val_df):,} (seed={params['seed']})")

    src_tokenizer, tgt_tokenizer = build_tokenizers(train_df, params)
    print(f"Source vocab size: {len(src_tokenizer):,}")
    print(f"Target vocab size: {len(tgt_tokenizer):,}")

    train_dataset = TranslationDataset(
        train_df,
        src_tokenizer,
        tgt_tokenizer,
        max_src_len=params["max_src_len"],
        max_tgt_len=params["max_tgt_len"],
    )
    val_dataset = TranslationDataset(
        val_df,
        src_tokenizer,
        tgt_tokenizer,
        max_src_len=params["max_src_len"],
        max_tgt_len=params["max_tgt_len"],
    )
    metric_val_dataset = TranslationDataset(
        metric_val_df,
        src_tokenizer,
        tgt_tokenizer,
        max_src_len=params["max_src_len"],
        max_tgt_len=params["max_tgt_len"],
    )
    collate_fn = make_collate_fn(src_tokenizer.pad_id, tgt_tokenizer.pad_id)

    train_loader = DataLoader(
        train_dataset,
        batch_size=params["batch_size"],
        shuffle=True,
        num_workers=params["num_workers"],
        pin_memory=params["pin_memory"],
        collate_fn=collate_fn,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=params["batch_size"],
        shuffle=False,
        num_workers=params["num_workers"],
        pin_memory=params["pin_memory"],
        collate_fn=collate_fn,
    )
    metric_val_loader = DataLoader(
        metric_val_dataset,
        batch_size=params["batch_size"],
        shuffle=False,
        num_workers=params["num_workers"],
        pin_memory=params["pin_memory"],
        collate_fn=collate_fn,
    )
    return train_loader, val_loader, metric_val_loader, src_tokenizer, tgt_tokenizer

In [ ]:
def save_checkpoint(path, checkpoint):
    # 임시 파일에 먼저 쓰고 교체해서, 저장 도중 런타임이 끊겨도 기존 파일이 깨지지 않게 함
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)  # 저장 폴더가 없으면 생성
    tmp_path = path.with_name(path.name + ".tmp")
    torch.save(checkpoint, tmp_path)
    tmp_path.replace(path)

In [ ]:
def train_model(device, params):
    torch.manual_seed(params["seed"])
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(params["seed"])

    train_loader, val_loader, metric_val_loader, src_tokenizer, tgt_tokenizer = prepare_data_loaders(params)
    # positional encoding은 src/tgt 양쪽에 쓰이므로 둘 중 더 긴 길이를 사용
    max_model_len = max(params["max_src_len"], params["max_tgt_len"])

    model = TranslationTransformer(
        src_vocab_size=len(src_tokenizer),
        tgt_vocab_size=len(tgt_tokenizer),
        src_pad_id=src_tokenizer.pad_id,
        tgt_pad_id=tgt_tokenizer.pad_id,
        d_model=params["d_model"],
        nhead=params["nhead"],
        num_encoder_layers=params["num_encoder_layers"],
        num_decoder_layers=params["num_decoder_layers"],
        dim_feedforward=params["dim_feedforward"],
        max_len=max_model_len,
        dropout=params["dropout"],
    ).to(device)

    print(f"\nTotal trainable parameters: {count_trainable_parameters(model):,}")
    print(
        "Decoding: "
        f"beam_size={params['beam_size']} | "
        f"length_penalty_alpha={params['length_penalty_alpha']}"
    )

    criterion = nn.CrossEntropyLoss(
        ignore_index=tgt_tokenizer.pad_id,
        label_smoothing=params["label_smoothing"],
        reduction="sum",
    )
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=params["learning_rate"],
        weight_decay=params["weight_decay"],
    )

    lr_lambda = get_transformer_lr_lambda(params["warmup_steps"])
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

    writer = SummaryWriter(params["tensorboard_dir"])

    # best model은 output_path, 이어서 학습용 마지막 상태는 <이름>_last.pt 에 저장
    output_path = Path(params["output_path"])
    last_path = output_path.with_name(f"{output_path.stem}_last{output_path.suffix}")
    best_bleu = -float("inf")

    for epoch in tqdm(range(params["epochs"])):
        train_loss = train_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            scheduler,
            device,
            grad_clip=params["grad_clip"],
        )

        val_loss, bleu, chrf, samples = evaluate_translation(
            model,
            val_loader,
            metric_val_loader,
            criterion,
            src_tokenizer,
            tgt_tokenizer,
            device,
            max_src_len=params["max_src_len"],
            max_tgt_len=params["max_tgt_len"],
            beam_size=params["beam_size"],
            length_penalty_alpha=params["length_penalty_alpha"],
            sample_count=params["sample_count"],
        )

        lr = optimizer.param_groups[0]["lr"]

        print(
            f"Epoch {epoch + 1}: train_loss={train_loss:.4f} "
            f"| val_loss={val_loss:.4f} | BLEU={bleu:.2f} | chrF={chrf:.2f} "
            f"| LR={lr:.8f}"
        )

        for idx, (src_text, ref_text, hyp_text) in enumerate(samples, start=1):
            print(f"  [{idx}] SRC: {src_text}")
            print(f"      REF: {ref_text}")
            print(f"      HYP: {hyp_text}")

        writer.add_scalars(
            "loss", {"train": train_loss, "validation": val_loss}, epoch + 1
        )
        writer.add_scalar("metrics/bleu", bleu, epoch + 1)
        writer.add_scalar("metrics/chrf", chrf, epoch + 1)
        writer.add_scalar("learning_rate", lr, epoch + 1)

        is_best = bleu > best_bleu
        if is_best:
            best_bleu = bleu

        # 모델 weight뿐 아니라 설정값과 tokenizer vocab도 같이 저장
        # tokenizer를 같이 저장해야 나중에 같은 ID 체계로 번역 가능
        # best_score, monitor 키는 진단 노트북(TF 05 - Translator v3)이 읽으므로 유지
        base_checkpoint = {
            "model_state_dict": model.state_dict(),
            "params": params,
            "src_tokenizer": src_tokenizer.to_dict(),
            "tgt_tokenizer": tgt_tokenizer.to_dict(),
            "best_score": best_bleu,
            "monitor": "bleu",
            "epoch": epoch + 1,  # 완료한 epoch 수
        }

        # best BLEU가 갱신된 epoch이면 best model을 바로 저장 (학습 도중 끊겨도 best가 남음)
        if is_best:
            save_checkpoint(output_path, base_checkpoint)
            print(f"  Saved best model (epoch {epoch + 1}): {output_path}")

        # 이어서 학습할 수 있도록 매 epoch 마지막 상태를 저장
        # optimizer/scheduler 상태가 있어야 LR 스케줄과 AdamW 모멘트가 끊김 없이 이어짐
        save_checkpoint(
            last_path,
            {
                **base_checkpoint,
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
            },
        )

    writer.close()

    print(f"Training finished. Best BLEU: {best_bleu:.2f}")
    print(f"Saved model: {output_path}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
train_model(device, PARAMS)

Device: cuda
Train rows: 2,394,058
Validation rows: 301,510
Metric evaluation rows: 1,000 (seed=42)

Building source SentencePiece tokenizer
Building target SentencePiece tokenizer

Source tokenizer
------------------------------------------------------------------------
vocab size: 12,000
special ids: pad=0, unk=1, bos=2, eos=3
max_len setting: 192
max observed length (+bos/eos): 1562
avg observed length (+bos/eos): 16.08
p95 observed length (+bos/eos): 34
truncated by max_len: 528 (0.02%)

Target tokenizer
------------------------------------------------------------------------
vocab size: 16,000
special ids: pad=0, unk=1, bos=2, eos=3
max_len setting: 160
max observed length (+bos/eos): 1532
avg observed length (+bos/eos): 17.67
p95 observed length (+bos/eos): 35
truncated by max_len: 1,044 (0.04%)
Source vocab size: 12,000
Target vocab size: 16,000


/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = TransformerEncoder(



Total trainable parameters: 66,684,544
Decoding: beam_size=4 | length_penalty_alpha=0.4


  0%|          | 0/50 [00:00<?, ?it/s]

Epoch 1: train_loss=5.3400 | val_loss=4.3188 | BLEU=8.91 | chrF=26.63 | LR=0.00009352
  [1] SRC: 한국 정부측과 연락해 총기를 구매해요.
      REF: They do this by communicating with their counterpart in Korea.
      HYP: It's a total of the Korean government and South Korea.
  [2] SRC: 이 계획은 많은 자금을 필요로 합니다.
      REF: This plan requires a lot of money.
      HYP: I need a lot of money.
  [3] SRC: 최근 UNDP 농업 프로젝트와 다른 UN 농식품 엔지니어링 프로젝트를 위해 정미소 공장 3세트를 계약한 바 있습니다.
      REF: Recently, we signed a three-set rice mill for UNDP agricultural projects and other UN agri-food engineering projects.
      HYP: Recently, we have recently developed a 3D project and a 3D model for the development of the company to establish a new project.
  [4] SRC: 다른 제품에 비해 1EV에서 3EV로 2 스탑 증가했습니다.
      REF: This is a 2 stop increase from 1EV to 3EV compared to other products.
      HYP: In 23, it has increased from 1 to 32 inches in the product.
  [5] SRC: 제가 기억하기로는 새로운 상품을 준비 중에 있으신 걸로 압니다.
      REF: As far as I remember, you ar

...

 94%|█████████▍| 47/50 [22:34:58<1:26:27, 1729.21s/it]

Epoch 48: train_loss=2.5214 | val_loss=2.6053 | BLEU=29.86 | chrF=55.93 | LR=0.00001493
  [1] SRC: 한국 정부측과 연락해 총기를 구매해요.
      REF: They do this by communicating with their counterpart in Korea.
      HYP: I am contacting the Korean government and purchasing firearms.
  [2] SRC: 이 계획은 많은 자금을 필요로 합니다.
      REF: This plan requires a lot of money.
      HYP: This plan requires a lot of funds.
  [3] SRC: 최근 UNDP 농업 프로젝트와 다른 UN 농식품 엔지니어링 프로젝트를 위해 정미소 공장 3세트를 계약한 바 있습니다.
      REF: Recently, we signed a three-set rice mill for UNDP agricultural projects and other UN agri-food engineering projects.
      HYP: We have recently signed a UND project to set up a 3D agricultural factory project for UNPN engineering projects and other agricultural products.
  [4] SRC: 다른 제품에 비해 1EV에서 3EV로 2 스탑 증가했습니다.
      REF: This is a 2 stop increase from 1EV to 3EV compared to other products.
      HYP: Compared to other products, it has increased from 12V to 3V.
  [5] SRC: 제가 기억하기로는 새로운 상품을 준비 중에 있으신 걸로 압니다.

 96%|█████████▌| 48/50 [23:03:49<57:39, 1729.55s/it]  

Epoch 49: train_loss=2.5180 | val_loss=2.6023 | BLEU=30.25 | chrF=56.28 | LR=0.00001477
  [1] SRC: 한국 정부측과 연락해 총기를 구매해요.
      REF: They do this by communicating with their counterpart in Korea.
      HYP: I am contacting the Korean government and purchasing firearms.
  [2] SRC: 이 계획은 많은 자금을 필요로 합니다.
      REF: This plan requires a lot of money.
      HYP: This plan requires a lot of funds.
  [3] SRC: 최근 UNDP 농업 프로젝트와 다른 UN 농식품 엔지니어링 프로젝트를 위해 정미소 공장 3세트를 계약한 바 있습니다.
      REF: Recently, we signed a three-set rice mill for UNDP agricultural projects and other UN agri-food engineering projects.
      HYP: Recently, we have signed a contract with another UND engineer for a three-year-old UPN agricultural plant engineering project and an agricultural food factory project.
  [4] SRC: 다른 제품에 비해 1EV에서 3EV로 2 스탑 증가했습니다.
      REF: This is a 2 stop increase from 1EV to 3EV compared to other products.
      HYP: Compared to other products, it has increased from 1 to 2VEV to 3V.
  [5] SRC: 제가 기억하

 98%|█████████▊| 49/50 [23:32:38<28:49, 1729.47s/it]

In [ ]:
# TensorBoard 결과 출력
%load_ext tensorboard
%tensorboard --logdir=../tensorboard/

In [ ]:
from google.colab import runtime
import time

time.sleep(30)
runtime.unassign()